In [1]:
import os
import pandas as pd

load_dir = r"C:\Users\olish\Documents\ml_projects\Steps_Predictor\model\data\clean\steps"

dfs = {}

for file_name in os.listdir(load_dir):
    if file_name.endswith(".pkl"):
        file_path = os.path.join(load_dir, file_name)
        
        df_name = file_name.replace(".pkl", "")
        dfs[df_name] = pd.read_pickle(file_path)

In [2]:
import numpy as np

data = dfs['oli']

data["month"] = data.date.dt.month.astype(str).astype("category")
data["time_idx"] = (data["date"] - data["date"].min()).dt.days
data["log_value"] = np.log(data.value + 1e-8)

data.head()

,date,value,month,time_idx,log_value
0,2018-11-18,4390,11,0,8.387085
1,2018-11-19,8335,11,1,9.028219
2,2018-11-20,6946,11,2,8.845921
3,2018-11-21,8350,11,3,9.030017
4,2018-11-22,11054,11,4,9.310548


In [3]:
print(data.loc[1500:1520])

           date  value month  time_idx  log_value
1500 2021-08-12   5648     8       998   8.639057
1502 2021-08-13  10236     8       999   9.233666
1504 2021-08-14   5843     8      1000   8.673000
1506 2021-08-15   6860     8      1001   8.833463
1508 2021-08-16   7827     8      1002   8.965335
1510 2021-08-17   5198     8      1003   8.556029
1514 2021-08-20   3031     8      1006   8.016648
1516 2021-08-21   9700     8      1007   9.179881
1518 2021-08-22   7560     8      1008   8.930626
1520 2021-08-23  10923     8      1009   9.298626


In [4]:
time_gap_df = data.sort_values(by="time_idx")

time_gap_df["time_diff"] = time_gap_df["time_idx"].diff()

gaps = time_gap_df[time_gap_df["time_diff"] > 1]

print(gaps)

           date  value month  time_idx  log_value  time_diff
1514 2021-08-20   3031     8      1006   8.016648        3.0


In [5]:
complete_date_range = pd.date_range(start=data["date"].min(), end=data["date"].max(), freq='D')

missing_dates = complete_date_range.difference(data["date"])

print(missing_dates)

DatetimeIndex(['2021-08-18', '2021-08-19'], dtype='datetime64[ns]', freq='D')


In [6]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting import GroupNormalizer

data["value"] = data["value"].astype(float)
data["log_value"] = data["log_value"].astype(float)

data["group"] = 0

max_prediction_length = 1
max_encoder_length = 30
training_cutoff = data["time_idx"].max() - max_prediction_length

training = TimeSeriesDataSet(
    data[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="value",
#     group_ids=["agency", "sku"],
    group_ids=["group"],
    min_encoder_length=max_encoder_length // 2,  
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
#     static_categoricals=["agency", "sku"],
#     static_reals=["avg_population_2017", "avg_yearly_household_income_2017"],
    time_varying_known_categoricals=["month"],
#     variable_groups={"special_days": special_days},  # group of categorical variables can be treated as one variable
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=[
        "value",
        "log_value",
#         "industry_volume",
#         "soda_volume",
#         "avg_max_temp",
#         "avg_volume_by_agency",
#         "avg_volume_by_sku",
    ],
#     target_normalizer=GroupNormalizer(
#         groups=["agency", "sku"], transformation="softplus"
#     ),  # use softplus and normalize by group
     target_normalizer=GroupNormalizer(
         groups=["group"], transformation="softplus"
         ),     
#
     add_relative_time_idx=True,
     add_target_scales=True,
     add_encoder_length=True,
     allow_missing_timesteps=True
)